# The vault and accounts

Platform secrets never live in the repository or in `.env`: `.env` holds
only the **link to a vault** (Bitwarden / Vaultwarden, HashiCorp Vault /
OpenBao, a KeePassXC file). Secrets are named `<platform>/<key>`, e.g.
`mastodon/token`. Here the `env` backend stands in for a vault: it reads
`AMOORION_SECRET_<PLATFORM>_<KEY>` variables, meant for development.

In [1]:
import os, tempfile

# Examples stay off your real setup: secrets from environment variables
# (the `env` vault) and an inbox in a temporary directory.
os.environ["AMOORION_VAULT"] = "env"
os.environ["AMOORION_INBOX"] = os.path.join(tempfile.mkdtemp(), "inbox.sqlite3")

In [2]:
from amoorion.vault import VaultLink, open_vault

link = VaultLink.from_env()
print(link)
vault = open_vault(link)

kind='env' addr=None token=None path=None


Every plugin says which secrets it needs. `accounts` tells which platforms
have them all, and never shows a value:

In [3]:
from amoorion.methods import Accounts

def ready():
    for status in Accounts(vault)():
        print(f"{status['platform']:16} {'ready' if status['ready'] else 'missing: ' + ', '.join(status['missing'])}")

ready()

blogger          missing: address, smtp-uri, smtp-password
bluesky          missing: username, password
discord          missing: token, channel
discord-webhook  missing: url
dryrun           ready
facebook         missing: page-id, page-token
mastodon         missing: uri, token
matrix           missing: uri, token, room
telegram         missing: token, chat
tumblr           missing: consumer-key, consumer-secret, token, token-secret, blog
x                missing: api-key, api-secret, access-token, access-secret


Give Mastodon its two secrets (made-up values) and ask again:

In [4]:
os.environ["AMOORION_SECRET_MASTODON_URI"] = "https://mastodon.example"
os.environ["AMOORION_SECRET_MASTODON_TOKEN"] = "not-a-real-token"
ready()

blogger          missing: address, smtp-uri, smtp-password
bluesky          missing: username, password
discord          missing: token, channel
discord-webhook  missing: url
dryrun           ready
facebook         missing: page-id, page-token
mastodon         ready
matrix           missing: uri, token, room
telegram         missing: token, chat
tumblr           missing: consumer-key, consumer-secret, token, token-secret, blog
x                missing: api-key, api-secret, access-token, access-secret


With a real vault, `.env` says where it is instead, e.g. for Bitwarden:

```sh
AMOORION_VAULT=bitwarden
AMOORION_VAULT_TOKEN=<the session of `bw unlock --raw`>
```

and the item `amoorion/mastodon` holds the fields `uri` and `token`. Each
platform's README lists its keys.